In [5]:
!pip install tensorflow keras numpy matplotlib scikit-learn

In [7]:
import pathlib

# 1) Set the main dataset directory
base_dir = pathlib.Path("Brain Tumor MRI Dataset")

# 2) Paths for training and testing folders
train_dir = base_dir / "Training"
test_dir = base_dir / "Testing"

print("Train dir :", train_dir)
print("Test dir: ", test_dir)


Train dir : Brain Tumor MRI Dataset\Training
Test dir:  Brain Tumor MRI Dataset\Testing


In [8]:
import tensorflow as tf

IMG_SIZE = (224, 224) # Size VGG16 expects
BATCH_SIZE = 32 # Number of images per step
SEED = 42 # For Reproducibility

### Explanation:

import tensorflow as tf loads TensorFlow so you can use its functions.​

IMG_SIZE = (224, 224) defines that every MRI image will be resized to 224×224 pixels (VGG16 input size).​

BATCH_SIZE = 32 tells TensorFlow to read 32 images at a time during training.​

SEED = 42 fixes the random seed so shuffling/splitting is repeatable.​

## Create training and validation datasets from Training

In [11]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,                   # path Brain Tumor MRI Dataset/Training
    labels="inferred",           # read labels from subfolder names
    label_mode="categorical",    # multi-class one-hot labels
    image_size=IMG_SIZE,         # Resize all images to 224x224
    batch_size=BATCH_SIZE,       # use 32 images per batch
    validation_split=0.2,        # 80% train, 20% validation
    subset="training",           # this dataset is the training part
    seed=SEED
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,                   # same folder, but now for validation
    labels="inferred",
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.2,        # same split ratio
    subset='validation',         # this dataset is the validation part
    seed=SEED
)

Found 5712 files belonging to 4 classes.
Using 4570 files for training.
Found 5712 files belonging to 4 classes.
Using 1142 files for validation.


### Explanation
train_dir tells TensorFlow to look inside Training where each class has its own folder (Glioma, Meningioma, etc.).​

labels="inferred" makes labels automatically based on those subfolder names.​

label_mode="categorical" returns labels as vectors (e.g., [0,0,1,0] for class 3) which is needed for multi‑class softmax.​

image_size=IMG_SIZE resizes every loaded image to 224×224.​

batch_size=BATCH_SIZE groups images into batches to train efficiently.​

validation_split=0.2 and subset="training"/"validation" tell TensorFlow to use 80% of images for training and 20% for validation, using the same random split controlled by seed=SEED.​

## Create test dataset from Testing

In [12]:
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    labels="inferred",
    label_mode="categorical",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 1311 files belonging to 4 classes.


## Create the VGG16 base

In [17]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models

# 1) Load pre-trained VGG16 without top classifier
base_model = VGG16(
    weights="imagenet",         # use weights learned on ImageNet
    include_top=False,          # remove original dense layer
    input_shape=(224, 224, 3)   # our image shape
)

# 2) Freeze base model weights so they don't change at first
base_model.trainable = False

### Explanation line by line:

from tensorflow.keras.applications import VGG16 imports the VGG16 architecture.​

from tensorflow.keras import layers, models imports building blocks for new layers and models.​

base_model = VGG16(...) creates the convolutional part of VGG16 with pretrained ImageNet weights and removes the original classifier (include_top=False) so it can be reused for your 4‑class task.​

input_shape=(224, 224, 3) matches the size you set when loading images.​

base_model.trainable = False freezes VGG16 so only your new layers will train initially (this makes training more stable on small medical datasets).​

## Add custom classification head

In [18]:
# 3) Build fulll model on top of VGG16
inputs = layers.Input(shape=(224, 224, 3))         # Input tensor for our images
x = base_model(inputs, training=False)             # pass image through VGG16
x = layers.GlobalAveragePooling2D()(x)             # reduce feature maps to a vector
x = layers.Dropout(0.5)(x)                         # dropout to reduce overfitting

outputs = layers.Dense(4, activation="softmax")(x) # 4 tumor classes
model = models.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)           │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ vgg16 (Functional)                   │ (None, 7, 7, 512)           │      14,714,688 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 512)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 4)                   │           2,052 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 14,716,740 (56.14 MB)

 Trainable params: 2,052 (8.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

## compile the model

In [19]:
from tensorflow.keras import optimizers, losses, metrics

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),            # how fasr to learn
    loss=losses.CategoricalCrossentropy(),                    # for multi-class labels
    metrics=[metrics.CategoricalAccuracy(name="accuracy")]    # report accuracy
)

## Train the Model

In [20]:
EPOCHS = 10

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

Epoch 1/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 594s 4s/step - accuracy: 0.2678 - loss: 6.7997 - val_accuracy: 0.3529 - val_loss: 2.6044
Epoch 2/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 574s 4s/step - accuracy: 0.3276 - loss: 5.0217 - val_accuracy: 0.5018 - val_loss: 1.7143
Epoch 3/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 643s 5s/step - accuracy: 0.3956 - loss: 4.0102 - val_accuracy: 0.6191 - val_loss: 1.2844
Epoch 4/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 24806s 175s/step - accuracy: 0.4606 - loss: 3.3374 - val_accuracy: 0.6821 - val_loss: 1.0727
Epoch 5/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 1035s 7s/step - accuracy: 0.5011 - loss: 2.7936 - val_accuracy: 0.7329 - val_loss: 0.9301
Epoch 6/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 568s 4s/step - accuracy: 0.5333 - loss: 2.4962 - val_accuracy: 0.7662 - val_loss: 0.8587
Epoch 7/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 2977s 21s/step - accuracy: 0.5764 - loss: 2.1320 - val_accuracy: 0.7916 - val_loss: 0.7863
Epoch 8/10
143/143 ━━━━━━━━━━━━━━━━━━━━ 568s 4s/step - accuracy: 0.6077 - loss: 1.9298 - v

In [21]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test accuracy:", test_acc)
print("Test loss:", test_loss)

41/41 ━━━━━━━━━━━━━━━━━━━━ 134s 3s/step - accuracy: 0.7742 - loss: 0.8669
Test accuracy: 0.7742181420326233
Test loss: 0.8669179677963257


In [22]:
from tensorflow.keras import Sequential
from tensorflow.keras import layers

data_augmentation = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

train_ds_aug = train_ds.map(
    lambda x,y: (data_augmentation(x, training=True), y)
)

In [23]:
# unfreeze last 4 layers of VGG16
for layer in base_model.layers[-4:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=5
)


Epoch 1/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 668s 5s/step - accuracy: 0.7311 - loss: 0.8052 - val_accuracy: 0.8616 - val_loss: 0.4229
Epoch 2/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 880s 6s/step - accuracy: 0.8254 - loss: 0.4755 - val_accuracy: 0.8923 - val_loss: 0.3333
Epoch 3/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 2021s 14s/step - accuracy: 0.8709 - loss: 0.3666 - val_accuracy: 0.9019 - val_loss: 0.2910
Epoch 4/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 868s 6s/step - accuracy: 0.8947 - loss: 0.2951 - val_accuracy: 0.9089 - val_loss: 0.2729
Epoch 5/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 653s 5s/step - accuracy: 0.9055 - loss: 0.2685 - val_accuracy: 0.9229 - val_loss: 0.2297


In [24]:
callback = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history_ft = model.fit(
    train_ds_aug,
    validation_data=val_ds,
    epochs=20,
    callbacks=[callback]
)

Epoch 1/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 871s 6s/step - accuracy: 0.9140 - loss: 0.2312 - val_accuracy: 0.9238 - val_loss: 0.2203
Epoch 2/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 1399s 10s/step - accuracy: 0.9304 - loss: 0.2082 - val_accuracy: 0.9308 - val_loss: 0.2156
Epoch 3/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 862s 6s/step - accuracy: 0.9333 - loss: 0.1870 - val_accuracy: 0.9273 - val_loss: 0.2114
Epoch 4/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 675s 5s/step - accuracy: 0.9462 - loss: 0.1478 - val_accuracy: 0.9256 - val_loss: 0.2362
Epoch 5/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 801s 6s/step - accuracy: 0.9389 - loss: 0.1582 - val_accuracy: 0.9370 - val_loss: 0.1986
Epoch 6/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 850s 6s/step - accuracy: 0.9554 - loss: 0.1268 - val_accuracy: 0.9387 - val_loss: 0.1658
Epoch 7/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 780s 5s/step - accuracy: 0.9571 - loss: 0.1204 - val_accuracy: 0.9405 - val_loss: 0.1856
Epoch 8/20
143/143 ━━━━━━━━━━━━━━━━━━━━ 2507s 18s/step - accuracy: 0.9637 - loss: 0.0987 - val_

In [25]:
test_loss_ft, test_acc_ft = model.evaluate(test_ds)
print("New test accuracy:", test_acc_ft)
print("New test loss:", test_loss_ft)

41/41 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - accuracy: 0.9657 - loss: 0.1021
New test accuracy: 0.9656750559806824
New test loss: 0.10213916003704071


In [26]:
model.save("app/brain_tumor_vgg16_ft.h5")